# 第33版：3秒FPS统计、100ms刷新、2小时运行
沿用第32版模型、硬件和实时矫正逻辑。


In [ ]:
from pathlib import Path
import os,sys
from IPython import get_ipython
if get_ipython() is not None:get_ipython().run_line_magic('xmode','Plain')
# ⭐ 2026-10-18：不依赖文件夹名字 —— 自动找【本版】目录
#   目录必须包含本版 RELEASE_ID，避免从V7等旧目录启动。
EXPECTED_RELEASE = 'p11-rgb-pspl-v9-ai-fps-20261101-33'
def _build_dir(p):
    # Match the declared identity, independently of renamed/nested folders.
    import ast
    p=Path(p)
    try:
        tree=ast.parse((p/'p11_contract.py').read_text(encoding='utf-8'))
        identity=next((ast.literal_eval(n.value) for n in tree.body
            if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='RELEASE_ID' for t in n.targets)),None)
        f=p/'v87_v74_tracking.py'
        return (identity==EXPECTED_RELEASE and f.is_file()
            and 'def _static_hold' in f.read_text(encoding='utf-8',errors='replace')
            and (p/'v97_album.py').is_file() and (p/'v97_1_button_left.png').is_file())
    except (OSError,SyntaxError,ValueError,TypeError):return False
BASE=Path.cwd().resolve()
if not _build_dir(BASE):
    root=Path('/home/xilinx/jupyter_notebooks')
    cands=[root]+[p.parent for p in root.glob('*/v87_v74_tracking.py')]+[p.parent for p in root.glob('*/*/v87_v74_tracking.py')]
    hit=sorted({p.resolve() for p in cands if _build_dir(p)},key=str)
    print('当前目录 =',BASE)
    print('找到的第18版目录 =',hit)
    if not hit:raise FileNotFoundError('未找到第18版完整目录。请在包含p11_contract.py和system.bit的第18版目录中运行本启动本；支持改名、重复嵌套和安装后缀。')
    if len(hit)>1:raise RuntimeError('存在多个第18版目录。请在你要使用的目录中打开本启动本，重启内核再运行，避免误用另一份程序：'+str(hit))
    BASE=hit[0]
else:
    print('当前目录就是第18版 =',BASE)
print('将使用：',BASE)
os.chdir(BASE)
sys.path.insert(0,str(BASE))
from p11_contract import verify_package,RELEASE_ID

# ⭐ 2026-10-24 C1：GIL 切换间隔。默认 5 ms ⇒ 主线程调完原生后要等最多 5 ms 才能重新拿到 GIL。
#   实测 input_pack_call_s 5.68 ms，而原生只 2.38 ms ⇒ 差的 3.3 ms 是等 GIL。
#   调小到 1 ms 能把这个等待上限压低；设 0 = 不改（回到默认）。
GIL_SWITCH_MS = 1.0
if GIL_SWITCH_MS:
    import sys as _sys
    _sys.setswitchinterval(GIL_SWITCH_MS / 1000.0)
import sys as _sys2
print('GIL switch interval = %.1f ms' % (_sys2.getswitchinterval() * 1000))

# ⭐ 2026-10-24 C1：CPU 亲和性。实测输入打包的 ctypes 调用 p50=5.68ms / p90=12.7ms，
#   而原生只 2.38ms；prepare 线程负载翻倍差值几乎不变 ⇒ 不是 GIL，是 OS 抢占。
#   1 = 主线程独占 CPU0、prepare/相 机线程绑 CPU1（双核三线程，减少互抢）；0 = 不绑。
CPU_AFFINITY = 0    # 2026-10-24 实测：绑核后 call p90 16.0/ service 44.0，全面变差 ⇒ 关闭
os.environ['P11_CPU_AFFINITY'] = '1' if CPU_AFFINITY else '0'
if CPU_AFFINITY:
    try:
        os.sched_setaffinity(0, {0})
        print('CPU_AFFINITY = 1  (主线程 -> CPU0)')
    except Exception as _e:
        print('CPU_AFFINITY 绑定失败:', _e)
verify_package(str(BASE/'system.bit'))

# ===== C2：YUYV 进 PL（2026-10-24）=====
# 1 = PL 侧做 YUV->RGB（PS 只 memcpy，DMA 只搬 2 B/px）
# 0 = 老路径（PS 转 BGRX32 再 memcpy，DMA 搬 4 B/px）
# **同一个比特流两种都支持** => 上板 A/B 只改这一行，不用换 bit。
# 注：老比特流没有格式位，读回 CTRL 判定后会自动退回 0（只会慢，不会错）。
YUYV_INPUT = 1
os.environ['P11_YUYV_INPUT'] = '1' if YUYV_INPUT else '0'
print('PL 侧 YUYV 转换 =', '开' if YUYV_INPUT else '关（老路径）')

# ⭐ 2026-10-25：逐帧日志降频。实测每帧落盘 7.7 KB、主线程深拷贝 4.4 ms（占墙钟 10%）。
#   1 = 每帧都记（要详细数据时用）；4 = 每 4 帧记一条（日常测帧率用，仍每秒 6~7 条）。
#   ⚠️ 只影响落盘记录；内存里的 rows 仍每帧全收，RUN_COMPLETED 的汇总不受影响。
LOG_EVERY = 4
os.environ['P11_LOG_EVERY'] = str(LOG_EVERY)
print('逐帧日志：每', LOG_EVERY, '帧记一条')
# ⭐⭐ 2026-10-18 合并 V1.1 优化：跳过 gate 失守后那次 _boundary 重搜
#   True  = 跳过（V1.1 / V89-M 已 A/B 采用为默认）
#   False = 保持 V94 原行为（做 A/B 对比时改这个）
# ⚠️ 环境变量，不是改 .py ⇒ 不触发 verify_package 校验，也不用重启内核。
import os
SKIP_RESEARCH = True           # ← 只改这一行
if SKIP_RESEARCH:
    os.environ['P11_SKIP_RESEARCH'] = '1'
else:
    os.environ.pop('P11_SKIP_RESEARCH', None)
print('SKIP_RESEARCH =', SKIP_RESEARCH)

# ⭐ 2026-10-18 静态保持：静止时跳过 observer/边测量，直接复用上一帧四角
#   True  = 开（预计静态 16.5 → 22~26 fps）
#   False = 原 V94-M 行为（A/B 用）
STATIC_HOLD = True
STATIC_EDGE_CONTINUITY = True  # V97.6 fixed physical edge reference
if STATIC_HOLD:
    os.environ['P11_STATIC_HOLD'] = '1'
else:
    os.environ.pop('P11_STATIC_HOLD', None)
print('STATIC_HOLD =', STATIC_HOLD)
os.environ['P11_STATIC_EDGE_CONTINUITY'] = '1' if STATIC_EDGE_CONTINUITY else '0'

# ⭐ 2026-10-18 静态输出冻结：保持帧不再跑 P1 remap/DMA，直接复用上一帧校正图
#   True  = 开（~26-30 fps，但静止时校正面板是"定格"的，看起来一卡一卡）
#   False = 关（~20-22 fps，但校正面板每帧更新、画面连续）← 默认
FREEZE_OUTPUT = False
if FREEZE_OUTPUT:
    os.environ['P11_FREEZE_OUTPUT'] = '1'
else:
    os.environ.pop('P11_FREEZE_OUTPUT', None)
print('FREEZE_OUTPUT =', FREEZE_OUTPUT)
# 可调参数（一般不用动）：进入/退出帧差、变化像素占比、最长保持秒、最少静止帧
os.environ['P11_HOLD_ENTER'] = '4.0'
os.environ['P11_HOLD_EXIT'] = '8.0'
os.environ['P11_HOLD_CHANGE'] = '0.03'
os.environ['P11_HOLD_EXIT_CHANGE'] = '0.08'
HOLD_MAX_S = 4.0   # 静态保持“强制完整重测”间隔(秒)。0.8 → 2.0 → 4.0。
                   # 判据：mode=lk_measured 的帧 tracker 中位 125ms，长段里仍占 6%，6→3% 预计 +1.3fps。
                   #       把静态帧率从 27 拖到 24；拉长间隔 DIRECTLY 少掉这些尖峰。
                   # 代价：长时间静止时慢速漂移纠正变慢。保守取 1.2，激进取 4.0。
os.environ['P11_HOLD_MAX_S'] = str(HOLD_MAX_S)
print('HOLD_MAX_S =', HOLD_MAX_S)
os.environ['P11_HOLD_MIN_FRAMES'] = '3'
PREPARE_AHEAD = 1   # PS 预取深度。1=默认(实测最优)；2=处理更快(service 33.7ms)但抢显示线程GIL，并排反而更卡。
                    # 判据：depth=2 时 PAIRED compose 39.4→45.2ms、显示 17→15fps（18:57 日志）。
os.environ['P11_PREPARE_AHEAD'] = str(PREPARE_AHEAD)
print('PREPARE_AHEAD =', PREPARE_AHEAD)
PREVIEW_SCALE = 'fused'   # 600P 预览缩放后端：fused(5:4多相,现状) / nearest(最近邻) / crop(居中裁,不缩放)
                          # 判据：600P 时 input_convert 中位 68.4ms、p90 94.6ms；480P 是 1:1 只 10.8ms。
                          # 三档差异只影响左边预览，校正输出/跟踪/光补全都不受影响。
os.environ['P11_PREVIEW_SCALE'] = PREVIEW_SCALE
print('PREVIEW_SCALE =', PREVIEW_SCALE)

from v75_ui import run_ui
OPTIONS=dict(device='/dev/video0',touch_device='auto',bitfile='system.bit',
    capture_dir='/home/xilinx/captures',capture_jpeg_quality=90,
    width=800,height=600,fps=15,output_size=(362,512),hdmi_mode=(1280,720),
    tracking_max_dim=400,detection_max_dim=640,hold_max_age_s=1.5,
    output_orientation='auto_two',hdmi_max_fps=30.,reacquire_budget_ms=420.,
    plan_strategy='fast_safe',proof_cpu_budget_ms=12.,progress_every=60,snapshot_timeout_s=20.,confirm_placement=False,
    book_rescue=False,book_rescue_cpu_ms=70.,
    auto_tracking_mode='strict',tracking_audit_s=1.0,
    outer_candidate_competition=False,recovery_local_first=False,
    detection_contrast_fallback=False,
    hybrid_auto_ai=True)
print(RELEASE_ID, 'V9 FPS33: 3s output FPS; 100ms update; 7200s session')


In [ ]:
# 顶部手动校准；放大阅读冻结显示但矫正继续；Interrupt退出整个应用。
result=run_ui(max_duration_s=7200.,max_frames=0,warmup_frames=5,**OPTIONS)
print('结束；日志:',result['run_log_path'])
